# 04B — Temporal Cross-Validation, Hyperparameter Tuning & XGBoost

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 4B — Robust Model Selection

## Why Phase 4B?

Phase 4A found a useful validation signal, but the Random Forest lost substantial ranking performance on the latest test period.

Rather than tuning against that test result, Phase 4B returns to the **development period only**:

`train + validation`

The original Phase 4A test split is excluded from tuning.

## Goals

- use expanding-window temporal cross-validation;
- compare model stability across time;
- tune Logistic Regression;
- tune Random Forest;
- tune HistGradientBoosting;
- add XGBoost;
- select by mean temporal-CV PR-AUC;
- use PR-AUC standard deviation as a stability tie-break;
- build out-of-fold predictions for threshold selection;
- save a tuned development candidate.

The resulting artifact is a **development candidate**, not a newly unbiased final holdout result.

## 1. Install XGBoost

In [1]:
# Run once from the project root if xgboost is not installed:
#
# uv add xgboost
#
# Then restart the notebook kernel if necessary.

import xgboost
print("xgboost:", xgboost.__version__)

xgboost: 3.4.1


## 2. Imports and project paths

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy project root chứa data/processed."
    )

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction


## 3. Load Phase 4B utilities

In [3]:
from src.features.build_features import TARGET_COLUMN
from src.models.temporal_cv import (
    fold_summary,
    make_expanding_window_folds,
)
from src.models.thresholds import (
    threshold_for_minimum_recall,
)
from src.models.tune import (
    cross_validate_config,
    load_development_data,
    refit_best_config,
    summarize_search,
)
from src.models.evaluate import find_best_f1_threshold
from src.models.tuning_registry import (
    get_tuning_configs,
)

## 4. Build the development dataset

Only the original Phase 3 train and validation partitions are combined.

The original test set is not loaded here.

In [4]:
development = load_development_data(
    PROJECT_ROOT
)

print("Development shape:", development.shape)
print(
    "Development range:",
    development["prediction_timestamp"].min(),
    "→",
    development["prediction_timestamp"].max(),
)
print(
    "Development late rate:",
    f"{development[TARGET_COLUMN].mean():.4%}",
)

assert len(development) == 67_515 + 14_467
assert development["order_id"].is_unique

Development shape: (81982, 44)
Development range: 2016-09-15 12:16:38 → 2018-06-21 11:59:03
Development late rate: 8.3811%


## 5. Expanding-window temporal CV

Design:

- first 50% of development rows → initial training window;
- remaining 50% → four ordered validation blocks;
- training window expands after each fold;
- validation always occurs strictly after training.

This is designed for irregular event-time orders while preserving chronology.

In [5]:
folds = make_expanding_window_folds(
    development,
    n_splits=4,
    initial_train_fraction=0.50,
)

folds_df = fold_summary(
    development,
    folds,
    target_column=TARGET_COLUMN,
)

display(folds_df)

for fold in folds:
    assert fold.train_end <= fold.validation_start

print("✓ Temporal fold chronology validated.")

,fold,train_rows,validation_rows,train_start,train_end,validation_start,validation_end,train_late_rate,validation_late_rate
0,1,40991,10248,2016-09-15 12:16:38,2017-12-13 09:58:26,2017-12-13 10:30:34,2018-02-03 20:50:00,0.0668,0.0625
1,2,51239,10248,2016-09-15 12:16:38,2018-02-03 20:50:00,2018-02-03 20:50:06,2018-03-19 19:15:28,0.0660,0.2074
2,3,61487,10248,2016-09-15 12:16:38,2018-03-19 19:15:28,2018-03-19 19:15:29,2018-05-03 17:54:18,0.0895,0.0754
3,4,71735,10247,2016-09-15 12:16:38,2018-05-03 17:54:18,2018-05-03 17:54:25,2018-06-21 11:59:03,0.0875,0.0579


✓ Temporal fold chronology validated.


## 6. Hyperparameter search space

In [6]:
configs = get_tuning_configs(
    quick=False
)

config_table = pd.DataFrame(
    [
        {
            "config_id": config.config_id,
            "model_family": config.model_family,
            **config.params,
        }
        for config in configs
    ]
)

print("Total configurations:", len(configs))
display(config_table)

Total configurations: 22


,config_id,model_family,C,class_weight,n_estimators,max_depth,min_samples_leaf,max_features,learning_rate,max_leaf_nodes,min_child_weight,subsample,colsample_bytree,reg_lambda,scale_pos_weight
0,logistic_regression_01,logistic_regression,0.1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,logistic_regression_02,logistic_regression,0.1000,balanced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,logistic_regression_03,logistic_regression,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,logistic_regression_04,logistic_regression,1.0000,balanced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,logistic_regression_05,logistic_regression,10.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,logistic_regression_06,logistic_regression,10.0000,balanced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,random_forest_01,random_forest,NaN,NaN,300.0000,12.0000,5.0000,sqrt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,random_forest_02,random_forest,NaN,NaN,300.0000,12.0000,20.0000,sqrt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,random_forest_03,random_forest,NaN,NaN,300.0000,NaN,5.0000,sqrt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,random_forest_04,random_forest,NaN,NaN,300.0000,NaN,20.0000,sqrt,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Run temporal cross-validation search

This is the computationally expensive section.

For a smoke test first, use the CLI:

`python -m src.models.tune --quick`

For the full Phase 4B search:

`python -m src.models.tune`

In [7]:
all_fold_metrics = []

for index, config in enumerate(configs, start=1):
    print(
        f"[{index:02d}/{len(configs):02d}] "
        f"{config.config_id}"
    )

    config_fold_metrics, _ = cross_validate_config(
        development,
        folds,
        config,
    )

    all_fold_metrics.append(
        config_fold_metrics
    )

fold_metrics = pd.concat(
    all_fold_metrics,
    ignore_index=True,
)

search_summary = summarize_search(
    fold_metrics,
    configs,
)

display(
    search_summary[
        [
            "config_id",
            "model_family",
            "mean_pr_auc",
            "std_pr_auc",
            "min_pr_auc",
            "max_pr_auc",
            "mean_roc_auc",
            "mean_recall",
            "mean_precision",
            "total_fit_seconds",
            "params",
        ]
    ].head(20)
)

[01/22] logistic_regression_01
[02/22] logistic_regression_02
[03/22] logistic_regression_03
[04/22] logistic_regression_04
[05/22] logistic_regression_05
[06/22] logistic_regression_06
[07/22] random_forest_01
[08/22] random_forest_02
[09/22] random_forest_03
[10/22] random_forest_04
[11/22] hist_gradient_boosting_01
[12/22] hist_gradient_boosting_02
[13/22] hist_gradient_boosting_03
[14/22] hist_gradient_boosting_04
[15/22] xgboost_01
[16/22] xgboost_02
[17/22] xgboost_03
[18/22] xgboost_04
[19/22] xgboost_05
[20/22] xgboost_06
[21/22] xgboost_07
[22/22] xgboost_08


,config_id,model_family,mean_pr_auc,std_pr_auc,min_pr_auc,max_pr_auc,mean_roc_auc,mean_recall,mean_precision,total_fit_seconds,params
0,xgboost_03,xgboost,0.2267,0.1145,0.1052,0.3738,0.7198,0.5133,0.1958,7.3694,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
1,xgboost_01,xgboost,0.2265,0.1114,0.1068,0.3663,0.7175,0.5583,0.1794,6.1638,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
2,xgboost_05,xgboost,0.2264,0.1123,0.1097,0.3729,0.7193,0.5777,0.1770,9.8034,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
3,xgboost_07,xgboost,0.2257,0.1135,0.1054,0.3731,0.7170,0.5083,0.1986,11.4719,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
4,xgboost_02,xgboost,0.2244,0.1044,0.1108,0.3565,0.7191,0.5827,0.1798,5.7717,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
5,xgboost_06,xgboost,0.2242,0.1056,0.1121,0.3619,0.7142,0.5827,0.1794,9.1471,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
6,hist_gradient_boosting_02,hist_gradient_boosting,0.2240,0.1133,0.1099,0.3752,0.7093,0.5222,0.1913,17.0736,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 31, ..."
7,xgboost_04,xgboost,0.2216,0.1089,0.1043,0.3622,0.7109,0.4967,0.2002,7.6334,"{""colsample_bytree"": 0.85, ""learning_rate"": 0...."
8,hist_gradient_boosting_01,hist_gradient_boosting,0.2210,0.1162,0.1080,0.3756,0.7127,0.5409,0.1782,18.3110,"{""learning_rate"": 0.05, ""max_leaf_nodes"": 15, ..."
9,hist_gradient_boosting_04,hist_gradient_boosting,0.2202,0.1188,0.1046,0.3797,0.7092,0.5277,0.1843,13.2447,"{""learning_rate"": 0.1, ""max_leaf_nodes"": 31, ""..."


## 8. Model-family summary

A model family should not be judged only by its best single fold.

Inspect both:

- mean PR-AUC;
- variation across temporal folds.

In [8]:
family_summary = (
    search_summary.groupby(
        "model_family",
        as_index=False,
    )
    .agg(
        best_mean_pr_auc=("mean_pr_auc", "max"),
        median_mean_pr_auc=("mean_pr_auc", "median"),
        best_min_fold_pr_auc=("min_pr_auc", "max"),
        configurations=("config_id", "count"),
    )
    .sort_values(
        "best_mean_pr_auc",
        ascending=False,
    )
)

display(family_summary)

,model_family,best_mean_pr_auc,median_mean_pr_auc,best_min_fold_pr_auc,configurations
3,xgboost,0.2267,0.2251,0.1121,8
0,hist_gradient_boosting,0.2240,0.2206,0.1099,4
1,logistic_regression,0.2186,0.2117,0.0859,6
2,random_forest,0.2117,0.2088,0.0984,4


## 9. Select best temporal-CV configuration

Selection rule is declared before any new holdout evaluation:

1. highest mean PR-AUC across temporal folds;
2. lower PR-AUC standard deviation as tie-break.

In [9]:
best_config_id = str(
    search_summary.iloc[0]["config_id"]
)

config_lookup = {
    config.config_id: config
    for config in configs
}

best_config = config_lookup[
    best_config_id
]

print("Best config:", best_config.config_id)
print("Family:", best_config.model_family)
print("Params:", best_config.params)
print(
    "Mean PR-AUC:",
    f"{search_summary.iloc[0]['mean_pr_auc']:.4f}",
)
print(
    "Std PR-AUC:",
    f"{search_summary.iloc[0]['std_pr_auc']:.4f}",
)

Best config: xgboost_03
Family: xgboost
Params: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 5, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 2.0, 'scale_pos_weight': 'auto'}
Mean PR-AUC: 0.2267
Std PR-AUC: 0.1145


## 10. Build out-of-fold probabilities for the winner

OOF probabilities are predictions on future blocks that were not used to fit each corresponding fold model.

They are appropriate for development-stage threshold analysis.

In [10]:
best_fold_metrics, best_oof = cross_validate_config(
    development,
    folds,
    best_config,
    collect_oof=True,
)

display(best_fold_metrics)

print("OOF rows:", len(best_oof))
print(
    "OOF range:",
    best_oof["prediction_timestamp"].min(),
    "→",
    best_oof["prediction_timestamp"].max(),
)

,config_id,model_family,fold,fit_seconds,train_rows,validation_rows,train_late_rate,validation_late_rate,pr_auc,roc_auc,precision,recall,f1,accuracy,positive_rate,threshold
0,xgboost_03,xgboost,1,1.6734,40991,10248,0.0668,0.0625,0.1052,0.6420,0.1075,0.3167,0.1605,0.7928,0.1842,0.5000
1,xgboost_03,xgboost,2,1.8958,51239,10248,0.0660,0.2074,0.3738,0.6926,0.3432,0.5675,0.4277,0.6851,0.3429,0.5000
2,xgboost_03,xgboost,3,2.1248,61487,10248,0.0895,0.0754,0.2499,0.7697,0.1038,0.8991,0.1861,0.4069,0.6533,0.5000
3,xgboost_03,xgboost,4,2.5011,71735,10247,0.0875,0.0579,0.1778,0.7749,0.2286,0.2698,0.2475,0.9050,0.0683,0.5000


OOF rows: 40991
OOF range: 2017-12-13 10:30:34 → 2018-06-21 11:59:03


## 11. Operating threshold analysis

Two threshold rules are retained:

### Best F1
A generic balance between precision and recall.

### Recall ≥ 50%
For a delivery-risk intervention system, missing most late orders may be unacceptable.

This second rule finds the threshold with the highest available precision while maintaining at least 50% recall on OOF development predictions.

In [11]:
best_f1_threshold = find_best_f1_threshold(
    best_oof["late_delivery"],
    best_oof["probability"],
)

recall_50_threshold = threshold_for_minimum_recall(
    best_oof["late_delivery"],
    best_oof["probability"],
    minimum_recall=0.50,
)

thresholds_df = pd.DataFrame(
    [
        {
            "rule": "best_oof_f1",
            "threshold": best_f1_threshold.threshold,
            "precision": best_f1_threshold.precision,
            "recall": best_f1_threshold.recall,
            "f1": best_f1_threshold.f1,
        },
        {
            "rule": recall_50_threshold.rule,
            "threshold": recall_50_threshold.threshold,
            "precision": recall_50_threshold.precision,
            "recall": recall_50_threshold.recall,
            "f1": recall_50_threshold.f1,
        },
    ]
)

display(thresholds_df)

,rule,threshold,precision,recall,f1
0,best_oof_f1,0.5650,0.1955,0.4412,0.2709
1,maximize_precision_subject_to_recall>=0.50,0.5260,0.1825,0.5002,0.2675


## 12. Refit the tuned development candidate

The selected configuration is now refitted on all development data:

`train + validation`

The original test set remains outside this fit.

In [12]:
best_pipeline = refit_best_config(
    development,
    best_config,
)

print(
    "✓ Best configuration refitted on",
    f"{len(development):,}",
    "development orders.",
)

✓ Best configuration refitted on 81,982 development orders.


## 13. Export Phase 4B results

The CLI implementation exports the complete artifacts.

Run from project root:

`python -m src.models.tune`

Artifacts:

- `models/best_tuned_candidate.joblib`
- `models/best_tuned_candidate_metadata.json`
- `reports/metrics/temporal_cv_folds.csv`
- `reports/metrics/temporal_tuning_fold_metrics.csv`
- `reports/metrics/temporal_tuning_summary.csv`
- `reports/metrics/best_tuned_oof_predictions.parquet`
- `reports/metrics/tuned_operating_thresholds.csv`

# 14. Conclusions

After Run All, use the actual temporal-CV results above as the source of truth.

## What Phase 4B changes

Phase 4A selected a model from one validation period.

Phase 4B instead measures how candidate configurations behave across multiple ordered historical periods.

This makes model selection more robust to the temporal variation already discovered in the Olist data.

## XGBoost

XGBoost is included as a strong gradient-boosted-tree baseline using:

- binary logistic objective;
- histogram tree method;
- AUC-PR evaluation metric;
- fold-specific automatic `scale_pos_weight`.

## Final selection rule

Choose the configuration with:

1. the highest mean temporal-CV PR-AUC;
2. lower temporal PR-AUC variation as a tie-break.

Do not choose a model because of its original Phase 4A test result.

## Thresholds

Keep threshold selection separate from ranking quality.

The notebook reports both:

- best OOF F1 threshold;
- highest-precision threshold satisfying Recall ≥ 50%.

The final operating threshold should eventually be tied to the business cost of interventions.

## Holdout caveat

The Phase 4A test set has already been observed and influenced the decision to conduct Phase 4B.

Therefore Phase 4B deliberately does not call a new score on that test set a pristine unbiased final estimate.

The tuned artifact is saved as a **development candidate**.

## Next step

After the full tuning run, inspect:

- model-family ranking;
- fold-by-fold PR-AUC;
- variance across time;
- OOF threshold trade-offs.

Then Phase 5 should perform model analysis, temporal error analysis, calibration and transparent holdout reporting.